# Networking & Port Publishing

## What's covered

- **The mental model** — every container gets its own network namespace, with its own interfaces, routes, iptables rules, and ports
- **The default `bridge` network** — what `docker0` is, why the default network is the *wrong* default
- **User-defined bridges** — better isolation, embedded DNS, the standard choice
- **Container-to-container DNS** — finding a peer by container name
- **The five built-in drivers** — `bridge`, `host`, `none`, `overlay`, `macvlan`/`ipvlan`
- **Publishing ports** — `-p host:container`, `-P`, the `EXPOSE` recap
- **What `-p` actually does** — DNAT, `iptables`, and the userland proxy
- **`host` network mode** — share the host's network stack, when it makes sense
- **`none`** — the air-gapped container
- **Inspecting networks** — `docker network ls`, `docker network inspect`, finding a container's IP
- **Connect / disconnect at runtime** — `docker network connect`, multi-network containers
- **DNS configuration** — `--dns`, `--add-host`, `/etc/resolv.conf` inside a container
- **Network aliases** — multiple names for the same container
- **`overlay` preview** — multi-host networking (full coverage in notebook 09 on Swarm)
- **`macvlan` / `ipvlan`** — when a container needs to look like a real device on your LAN

By the end you should be able to wire two containers together, expose only the ports you mean to, and explain *exactly* what happens to a packet between a browser and your containerised web app.

## The mental model

When a container starts on a non-`host`-mode network, the kernel gives it a **network namespace** of its own. That namespace has:

- Its own set of network interfaces (typically just `lo` and a virtual `eth0`)
- Its own IP address(es)
- Its own routing table
- Its own `iptables` / `nftables` rules
- Its own set of open ports

From inside the container, this looks like a tiny, isolated machine on a network. From the host, the container's `eth0` is one end of a virtual ethernet pair (`veth`); the other end is plugged into a Linux bridge on the host (`docker0` for the default network, or a `br-xxxxx` for user-defined networks).

```
                                                Host
                              +---------------------------------------+
                              |                                       |
   Container A                |   docker0 (bridge, 172.17.0.1)        |
   +-----------+ veth-pair    |        |          |          |        |
   | eth0      |==============|========+          |          |        |
   | 172.17.0.2|              |                   |          |        |
   +-----------+              |                   |          |        |
                              |   Container B   veth          |       |
                              |   +-----------+   |           |       |
                              |   | eth0      |===+           |       |
                              |   | 172.17.0.3|               |       |
                              |   +-----------+               |       |
                              |                                       |
                              |   eth0 (the host's real NIC) 10.0.0.5 |
                              +---------------------------------------+
                                                |
                                                v
                                          your LAN / internet
```

The bridge (`docker0`) is a Layer-2 switch in software. Containers on the same bridge can talk to each other directly. Reaching outside the host requires NAT, which the daemon configures on `docker0`'s upstream gateway.

Three big consequences:

1. **`localhost` inside a container is the container.** It is **not** the host. To reach the host from a container, you use a real IP (or the magic name `host.docker.internal` on Docker Desktop).
2. **Containers on the same user-defined bridge can reach each other by container name.** The daemon runs an embedded DNS resolver inside each container's namespace.
3. **Inbound traffic from the outside world has to be deliberately routed in.** That's what `-p` does.

## The default `bridge` network

When you install Docker, you get a network called `bridge` (driver: `bridge`) and a Linux bridge called `docker0` on the host. Every container started without `--network` lands here.

```bash
docker network ls
# NETWORK ID     NAME      DRIVER    SCOPE
# 7c8e3c2a...    bridge    bridge    local
# a9f1d8e2...    host      host      local
# 1f4a7b5d...    none      null      local
```

This default `bridge` works, but it has two annoying limitations:

1. **No embedded DNS.** Containers on the default bridge can talk to each other only by IP. `docker run --name web ...` and `docker run --name api ...` won't be able to resolve `api` as a hostname from inside `web`. You have to find the IP with `docker inspect`.
2. **Less isolation.** Every container without an explicit `--network` joins the same default bridge. So a container you just ran for an unrelated test can talk to your dev Postgres on the same bridge.

For these reasons, **use a user-defined bridge instead.** The default `bridge` is mostly a historical default; don't rely on it for anything you care about.

## User-defined bridges — the standard choice

Creating a user-defined bridge takes one command:

```bash
docker network create app-net
```

What you get versus the default bridge:

| Feature | Default `bridge` | User-defined bridge |
|---|---|---|
| Embedded DNS | No | **Yes** — resolve other containers by `--name` |
| Isolation | Shared with every other "no `--network`" container | Only containers attached to this network |
| Attach / detach at runtime | Awkward | `docker network connect/disconnect NETWORK CONTAINER` |
| MTU, gateway, subnet | Daemon defaults | Per-network configuration |

The standard pattern: one user-defined bridge per "application unit" (a web service + its DB + its cache), and attach all containers in that unit to it.

```bash
docker network create app-net
docker run -d --name db   --network app-net  postgres:16
docker run -d --name api  --network app-net  --env DB_HOST=db myorg/api
```

Inside the `api` container, the hostname `db` resolves to the IP of the `db` container. No port mapping needed between them — they're on the same bridge, talking on their container ports.

In [ ]:
# Create a user-defined bridge and demonstrate container-by-name DNS
!docker network create app-net 2>/dev/null || true
!docker run -d --name redis-demo --network app-net redis:7-alpine
!sleep 1
!docker run --rm --network app-net redis:7-alpine redis-cli -h redis-demo PING
!docker rm -f redis-demo
!docker network rm app-net

## The five built-in drivers

Docker ships with a handful of network drivers. Five matter.

| Driver | What it does | When to use |
|---|---|---|
| **`bridge`** | The default. Each container gets a private IP on a Linux bridge on the host. Outbound NAT'd. Inbound only via `-p`. | 99% of single-host work. |
| **`host`** | The container shares the **host's** network namespace. No isolation. The container's processes bind directly to the host's interfaces. | When you need maximum network performance, or to use raw sockets / `iptables` from inside the container. |
| **`none`** | The container gets a network namespace with only `lo`. No external connectivity at all. | Sandbox / forensic analysis where you explicitly want zero network. |
| **`overlay`** | Spans multiple hosts. Containers on different physical machines can see each other on the same virtual network. Built on VXLAN. | Docker Swarm clusters; covered in notebook 09. |
| **`macvlan`** / **`ipvlan`** | The container gets a MAC address (or just an IP) on the host's physical LAN. Looks like a real machine to the rest of the network. | Edge cases — legacy apps that need to broadcast on the LAN, IoT bridges, security tools. Not recommended for general use. |

There are also third-party network plugins (Calico, Weave, Cilium), mostly relevant in Kubernetes. For plain Docker, the five built-ins cover almost everything.

## Publishing ports

A container's processes bind to ports inside their network namespace. By default those ports are **not reachable from outside the host** — the bridge is private. To let outside traffic in, you have to **publish** a port.

### `-p host:container`

```bash
docker run -d -p 8080:80 nginx:alpine
```

Reads: "take whatever arrives on the host's port 8080 and forward it to the container's port 80." Now `curl http://localhost:8080` hits the Nginx inside the container.

Variations:

- **`-p 8080:80`** — bind on all host interfaces (`0.0.0.0:8080`).
- **`-p 127.0.0.1:8080:80`** — bind only on localhost. The container is reachable from your laptop but not from the wider network. **Use this for any dev service that doesn't need to be remote-accessible** — much safer.
- **`-p 8080:80/udp`** — pick the protocol explicitly. Default is TCP.
- **`-p 80`** (host port omitted) — bind to a random free port on the host. Read it back with `docker port CONTAINER` or the `PORTS` column of `docker ps`.

### `-P` (capital P) — auto-publish everything `EXPOSE`d

`-P` (no value) publishes every port the image's `EXPOSE` instruction declared, each to a **random** host port. Convenient for testing, almost never what production wants.

### `EXPOSE` is just documentation

Recall from notebook 02: `EXPOSE 80` in a Dockerfile is **metadata**. It does *not* publish the port. It just hints "this image listens on 80." `-P` reads that hint to know what to expose; humans read it to know what to map. Without an `EXPOSE` you can still `-p 8080:80` — `EXPOSE` is hint, not gate.

## What `-p` actually does under the hood

When you run `docker run -p 8080:80 nginx`, the daemon does two things:

1. **`iptables` DNAT rule.** On Linux, it inserts a rule in the `nat` table's `PREROUTING` chain that translates incoming traffic to `host:8080` into traffic destined for `container-ip:80`. You can see this with `sudo iptables -t nat -L DOCKER -n`. This is the fast path — kernel-space, zero-copy.
2. **Userland proxy fallback.** For traffic that *can't* go through the iptables rule (e.g. localhost-to-localhost, where DNAT skips), the daemon spawns a small process (`docker-proxy`) that listens on `host:8080` and forwards to `container:80`. This is the slow path — a syscall round-trip per packet.

You can disable the userland proxy in `daemon.json` (`"userland-proxy": false`) if you want pure iptables. Most teams leave it on.

### Why this matters

- **Performance.** Heavy network workloads should test `-p`'s impact. For most apps the cost is invisible.
- **Visibility.** When two containers can't talk over a published port from inside the host, the iptables rule is the suspect. `sudo iptables -L -n -v` is your debug tool.
- **The "I can't bind that port" error.** If something on the host is already listening on `:8080`, `-p 8080:80` will fail. Pick a different host port.
- **macOS / Windows reality.** On Docker Desktop, there's no real `iptables` chain on your Mac — Desktop emulates port publishing by listening on macOS-level ports in its VM-bridge. Same flag, different mechanism.

## `host` network mode — share the host's stack

`--network host` skips all of this. The container has **no** namespace of its own; it sees and binds the host's interfaces directly.

```bash
docker run --network host nginx
# nginx is now listening on the host's port 80, not a container port
```

What you give up:

- **No port mapping.** `-p` is ignored. You bind directly.
- **No isolation.** If two `host`-network containers both want port 80, the second fails.
- **No bridge.** No `docker0`, no embedded DNS.

What you gain:

- **No NAT overhead.** Useful for very-high-throughput servers (load balancers, packet inspection).
- **Raw access to the host's network configuration.** Some tools need this.
- **Simpler debugging.** `netstat`, `tcpdump`, `iptables` inside the container all see exactly what the host sees.

Trade-off: **`host` mode breaks the whole "container is isolated" promise** for the network specifically. Use it sparingly and only when you have a specific reason.

**Note on Desktop.** `--network host` works on Linux. On Docker Desktop (Mac/Windows) it's only partially supported — it shares the *VM's* network, not your laptop's. Don't rely on it cross-platform.

## `none` — no network

```bash
docker run --rm --network none alpine ip addr
```

The container's namespace contains only `lo`. No `eth0`, no DNS, no outbound. Useful for sandboxed batch jobs that should never phone home, or forensic analysis of an image.

You can later `docker network connect` a network to add interfaces, so `--network none` isn't necessarily permanent for the container's lifetime.

## Inspecting networks

A few commands you'll run constantly.

```bash
docker network ls                    # all networks
docker network inspect app-net       # full JSON: subnet, gateway, containers attached
docker network inspect app-net -f '{{range .Containers}}{{.Name}} {{.IPv4Address}}{{println}}{{end}}'
docker inspect CONTAINER             # includes .NetworkSettings with attached networks
docker port CONTAINER                # published port mappings
```

To get a single container's IP on a network:

```bash
docker inspect -f '{{.NetworkSettings.Networks.app-net.IPAddress}}' my-container
```

A container attached to multiple networks has multiple IPs — one per network.

In [ ]:
# Inspect-flow demo
!docker network create demo-net 2>/dev/null || true
!docker run -d --name net-demo --network demo-net -p 127.0.0.1:8091:80 nginx:alpine
!sleep 1
!echo '--- networks attached ---'
!docker inspect -f '{{range $k, $v := .NetworkSettings.Networks}}{{$k}}={{$v.IPAddress}} {{end}}' net-demo
!echo '--- published ports ---'
!docker port net-demo
!echo '--- network listing ---'
!docker network ls --filter name=demo-net
!docker rm -f net-demo
!docker network rm demo-net

## Connect / disconnect at runtime

A container can be attached to multiple networks at once and you can change the attachments without restarting.

```bash
docker network create web-net
docker network create db-net

docker run -d --name api --network web-net myorg/api
docker network connect db-net api   # now api is on BOTH web-net and db-net
docker network disconnect web-net api   # now only on db-net
```

Two common uses:

- **Tiered apps.** A reverse proxy on `web-net` + a database on `db-net`. The API server attaches to both. The proxy can't see the DB; the DB isn't reachable from the public side.
- **Adding a debugger.** Attach an existing running container to a temporary inspection network with a `wireshark` or `tcpdump` sidecar.

The `--network` flag at `docker run` only takes one network. To attach to more than one at startup, run with one, then `docker network connect` the rest before the app starts caring.

## DNS configuration

By default a container's `/etc/resolv.conf` is generated by the daemon: it lists `127.0.0.11` (Docker's embedded resolver) for user-defined bridges and the host's upstream DNS for everything else.

The embedded resolver:

- Resolves other container names on the same user-defined network → their IPs
- Resolves any `--network-alias` you've assigned
- Forwards everything else to the host's upstream DNS

You can override:

- **`--dns 1.1.1.1`** — set a custom DNS server. Overrides the daemon default.
- **`--dns-search example.com`** — search domain.
- **`--add-host db.internal:10.0.0.5`** — static `/etc/hosts` entry inside the container. Useful for "I want this name to resolve to this IP without DNS."
- **`--hostname my-host`** — set the container's hostname (what `hostname` returns inside).
- **`--network-alias web,frontend`** — alternate names this container can be reached by on its network. Multiple containers can share an alias to round-robin between them.

```bash
docker run -d --network app-net --network-alias api --network-alias api-v1 myorg/api:1.0
# Other containers on app-net can `curl http://api` OR `curl http://api-v1`
```

## `overlay` — multi-host networks (preview)

Everything above is **single-host** networking — all containers on one Docker daemon. As soon as you have two or more hosts and want containers on host A to reach containers on host B as if they were on the same LAN, you need an **overlay network**.

`overlay` uses VXLAN under the hood: each cross-host packet is wrapped in a UDP datagram that traverses the underlay (the physical network between hosts) and is unwrapped on arrival. Hosts in the cluster gossip about which container lives where; routing tables stay in sync.

You can only create overlay networks **inside a Swarm cluster** (or via the experimental `docker network create -d overlay --attachable` for standalone use). Notebook 09 covers Swarm and overlay networks together, end to end.

For now just know: `overlay` is how Swarm services on different machines talk; if you're not in a Swarm, you won't reach for it.

## `macvlan` / `ipvlan` — real LAN IPs

`macvlan` gives the container a **separate MAC address** on the host's physical LAN. To the rest of the network — switches, routers, DHCP servers — the container looks like a wholly distinct machine. It has a real LAN IP. No NAT.

`ipvlan` is similar but shares the host's MAC, giving the container an IP without a unique MAC. Useful when your switch rejects unknown MACs.

When you'd want this:

- A containerised legacy app that broadcasts on the LAN (DLNA, mDNS, old industrial protocols).
- IoT or home-automation bridges where the container needs to participate in LAN discovery.
- Network appliances (IDS sensors, packet capture) that need promiscuous mode.

When you wouldn't:

- Pretty much anything else. `bridge` + `-p` is simpler, more portable, and doesn't depend on your physical network.

```bash
docker network create -d macvlan \
  --subnet=192.168.1.0/24 \
  --gateway=192.168.1.1 \
  -o parent=eth0 \
  lan-net

docker run -d --network lan-net --ip=192.168.1.250 myorg/legacy-app
```

Be aware: by default, **the host itself can't reach a macvlan container** (the kernel won't loop traffic back through the parent interface). The standard workaround is to create a `macvlan` sub-interface on the host as well, then route through it.